# 📬 Notebook 1: Queues vs Pub/Sub — what's the difference?

When two pieces of software talk to each other, they often go through a **message broker** instead of calling each other directly. Two of the most common patterns are:

- 🟦 **Queue (point-to-point)** — one producer, many workers, and **only one** worker handles each message. Used for "do this job" workloads.
- 🟪 **Pub/Sub (topic)** — one publisher, many subscribers, and **every** subscriber gets a copy of every message. Used for "tell everyone this happened" workloads.

In this notebook we implement both with nothing but Python's standard library, so the mechanics are obvious.

## Learning objectives
- Build a tiny in-memory queue and observe load-balancing across workers.
- Build a tiny in-memory pub/sub bus and observe fan-out to all subscribers.
- Know when to pick which.

## 🛠️ Setup

```bash
cd 01-foundations/messaging-basics
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). Reload the window if the kernel doesn't appear: `Cmd+Shift+P` → **Reload Window**.

This notebook uses only Python's stdlib (`queue`, `threading`).

## 🟦 Approach 1: Queue (one consumer per message)

Three workers share the same queue. Each message is delivered to **exactly one** of them — whichever pulls first wins. This naturally **load-balances** work.

In [ ]:
import queue
import threading
import time
import random

q = queue.Queue()
results = {}  # worker_id -> list of jobs it handled
results_lock = threading.Lock()

def worker(worker_id):
    while True:
        job = q.get()
        if job is None:        # poison pill = shutdown signal
            return
        time.sleep(random.uniform(0.01, 0.05))
        with results_lock:
            results.setdefault(worker_id, []).append(job)
        q.task_done()

# Start 3 workers
threads = [threading.Thread(target=worker, args=(i,), daemon=True) for i in range(3)]
for t in threads: t.start()

# Producer puts 10 jobs in the queue
for i in range(10):
    q.put(f"job-{i}")

q.join()  # wait for all jobs to be done
for _ in threads: q.put(None)  # tell workers to shut down
for t in threads: t.join()

for w, jobs in sorted(results.items()):
    print(f"worker {w} handled {len(jobs):2d} jobs: {jobs}")
print("\nNotice: each job was handled by exactly ONE worker.")

## 🟪 Approach 2: Pub/Sub (every subscriber gets a copy)

Now we want a different shape: when something interesting happens, we want **every** interested service to find out. Each subscriber gets its own private queue, and the publisher fans the message out to all of them.

In [ ]:
class PubSub:
    def __init__(self):
        self.subs: dict[str, list[queue.Queue]] = {}

    def subscribe(self, topic: str) -> queue.Queue:
        inbox = queue.Queue()
        self.subs.setdefault(topic, []).append(inbox)
        return inbox

    def publish(self, topic: str, message):
        # Deliver a copy to EVERY subscriber on the topic.
        for inbox in self.subs.get(topic, []):
            inbox.put(message)

bus = PubSub()
analytics = bus.subscribe("user.signup")
welcome_email = bus.subscribe("user.signup")
audit_log = bus.subscribe("user.signup")

bus.publish("user.signup", {"user": "alice"})
bus.publish("user.signup", {"user": "bob"})

for name, inbox in [("analytics", analytics),
                    ("welcome_email", welcome_email),
                    ("audit_log", audit_log)]:
    received = []
    while not inbox.empty():
        received.append(inbox.get())
    print(f"{name:14s} got {received}")

print("\nNotice: every subscriber received EVERY message.")

## 🤔 When to use which?

| | Queue | Pub/Sub |
|---|---|---|
| Each message handled by | exactly one worker | every subscriber |
| Good for | background jobs, work distribution | event broadcasting |
| Examples | image resizing, sending emails | "user signed up", "order placed" |
| Real-world tools | SQS, RabbitMQ queues, Redis lists | Kafka topics, Redis pub/sub, NATS |

You will often see **both** in the same system: an event is *published*, and one of the subscribers is itself a queue feeding workers.